# Basic FastHTML HTMX SSE Integration Example

> Demonstrates real-time tqdm progress tracking in FastHTML using HTMX with the SSE extension. This approach combines the efficiency of Server-Sent Events with HTMX's declarative style for automatic DOM updates. Shows how to capture tqdm output from background tasks and stream live progress updates without custom JavaScript.

In [1]:
from fasthtml.common import *
import uuid, time
import asyncio

from cjm_tqdm_capture.progress_monitor import ProgressMonitor
from cjm_tqdm_capture.job_runner import JobRunner
from cjm_tqdm_capture.streaming import sse_stream_async

In [2]:
# For Jupyter display
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from IPython.display import display

# Import DaisyUI factories
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_behaviors
from cjm_fasthtml_daisyui.components.feedback.progress import progress, progress_colors

# Import Tailwind factories
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.typography import font_weight, font_size
from cjm_fasthtml_tailwind.utilities.sizing import w
from cjm_fasthtml_tailwind.core.base import combine_classes

In [3]:
# Create app
app, rt = create_test_app(theme=DaisyUITheme.LIGHT)

# Initialize monitor and runner
monitor = ProgressMonitor()
runner = JobRunner(monitor)

In [4]:
# Add HTMX SSE extension after HTMX script
def get_htmx_idx(hdrs):
    return next((i for i, hdr in enumerate(hdrs) if (hdr.attrs.get('src') or '').endswith('htmx.min.js')), -1)

htmx_idx = get_htmx_idx(app.hdrs)
if htmx_idx >= 0:
    # Insert SSE extension right after HTMX
    app.hdrs.insert(htmx_idx+1, Script(src="https://unpkg.com/htmx-ext-sse@2.2.3/dist/sse.min.js"))
else:
    print("HTMX not found")

In [5]:
# Define a simple task that uses tqdm
def simple_task():
    from tqdm import tqdm
    import time
    
    results = []
    for i in tqdm(range(100), desc="Processing"):
        time.sleep(0.01)
        results.append(i * 2)
    return results

In [6]:
# Create helper function for Progress bars
def create_progress_bar(value="0", max="100", **attrs):
    """Create a styled progress bar with consistent styling"""
    return Progress(
        value=str(value), 
        max=str(max), 
        cls=combine_classes(progress, progress_colors.primary, w.full),
        **attrs
    )

# Create helper function for Progress labels
def create_progress_label(text="Progress:", **attrs):
    """Create a styled progress label with consistent styling"""
    return P(text, cls=combine_classes(font_weight.bold), **attrs)

# Create helper function for status messages
def create_status_message(text, **attrs):
    """Create a styled status message with consistent styling"""
    return P(text, cls=combine_classes(m.t(2), font_size.sm), **attrs)

# Create helper function for complete progress display with SSE
def create_sse_progress_display(job_id, initial_value="0", initial_status="Starting..."):
    """Create a progress display connected to SSE stream"""
    print(f"[CREATE_SSE] Creating SSE progress display for job_id: {job_id}")
    
    # Create the display with proper SSE handling for the close event
    display = Div(
        create_progress_label(id="progress-label"),
        create_progress_bar(value=initial_value, id="progress-bar"),
        create_status_message(initial_status, id="status-text"),
        # Add a script to log when SSE connects
        Script(f"""
            console.log('[CLIENT] SSE progress display created for job_id: {job_id}');
        """),
        # HTMX SSE attributes
        hx_ext="sse",
        sse_connect=f"/stream?job_id={job_id}",
        sse_swap="message",
        hx_swap="innerHTML",
        id="progress-display",
        # Handle the close event by swapping its content
        **{"sse-close": "close"}
    )
    
    print(f"[CREATE_SSE] SSE attributes: sse-connect=/stream?job_id={job_id}")
    return display

# Helper to render start button with appropriate state
def render_start_button(disabled=False):
    """Render start button with appropriate state"""
    btn_classes = combine_classes(
        btn, 
        btn_colors.primary,
        btn_behaviors.disabled if disabled else ""
    )
    
    return Button(
        "Start Task", 
        id="start-btn",
        hx_post="/start_task" if not disabled else None,
        hx_target="#progress-container",
        hx_swap="innerHTML",
        cls=btn_classes,
        disabled=disabled
    )

In [7]:
@rt("/")
def index():
    return create_test_page(
        "Basic HTMX SSE Progress Demo",
        Div(
            H2("Simple Progress Tracking with HTMX SSE Extension"),
            render_start_button(disabled=False),
            Div(
                P("Ready", cls=combine_classes(font_weight.bold, m.t(4))),
                id="progress-container",
                cls=str(m.t(4))
            ),
            cls=str(p(8))
        )
    )

In [8]:
# API endpoints using HTMX SSE
@rt("/start_task")
def start_task():
    job_id = str(uuid.uuid4())
    print(f"[START_TASK] Starting job with ID: {job_id}")
    
    runner.start(
        job_id, 
        simple_task,
        patch_kwargs={
            "min_delta_pct": 5,
            "min_update_interval": 0.05,
            "emit_initial": True
        }
    )
    
    print(f"[START_TASK] Job {job_id} started, returning SSE progress display")
    
    # Return SSE-connected progress display and disable button
    return Div(
        # Disable button via JavaScript (since we need immediate feedback)
        Script("""
            console.log('[CLIENT] Disabling start button');
            var startBtn = document.getElementById('start-btn');
            if (startBtn) {
                startBtn.disabled = true;
                startBtn.classList.add('btn-disabled');
                startBtn.removeAttribute('hx-post');
            }
            console.log('[CLIENT] About to connect to SSE');
        """),
        # Progress display with SSE connection
        create_sse_progress_display(job_id)
    )

@rt("/stream")
def stream(job_id: str):
    """SSE endpoint that streams progress updates as HTML"""
    
    async def html_stream():
        """Generate HTML updates for HTMX SSE"""
        import json
        
        message_count = 0
        
        try:
            # Use the async SSE stream generator
            async for data in sse_stream_async(
                monitor, 
                job_id, 
                interval=0.1,
                heartbeat=30.0,  # Increased heartbeat interval
                wait_for_start=True,
                start_timeout=5.0
            ):
                
                # Parse the SSE data format
                if data.startswith('data: '):
                    try:
                        # Extract JSON from SSE data line
                        json_str = data[6:].strip()
                        if json_str and json_str != '{}':
                            progress_data = json.loads(json_str)
                            message_count += 1
                            
                            
                            # Create HTML content based on progress
                            if progress_data.get('completed'):
                                # Final state - re-enable button and show completion
                                html_content = Div(
                                    create_progress_label("Progress:", id="progress-label"),
                                    create_progress_bar(value="100", id="progress-bar"),
                                    create_status_message("Completed!", id="status-text"),
                                    Script("""
                                    console.log('[CLIENT] Task completed, closing SSE');
                                    setTimeout(function() {
                                        // Re-enable the start button
                                        var startBtn = document.getElementById('start-btn');
                                        if (startBtn) {
                                            startBtn.disabled = false;
                                            startBtn.classList.remove('btn-disabled');
                                            startBtn.setAttribute('hx-post', '/start_task');
                                            htmx.process(startBtn);
                                            console.log('[CLIENT] Re-enable button.');
                                        }
                                        // Close the SSE connection
                                        var progressDisplay = document.getElementById('progress-display');
                                        if (progressDisplay && progressDisplay.sseEventSource) {
                                            console.log('[CLIENT] Closing SSE EventSource');
                                            progressDisplay.sseEventSource.close();
                                        }
                                    }, 100);
                                """)
                                )
                                
                                # Send final update
                                yield sse_message(html_content)
                                
                                # Send a custom event to trigger close and re-enable button
                                close_script = Script("""
                                    console.log('[CLIENT] Task completed, closing SSE');
                                    setTimeout(function() {
                                        // Re-enable the start button
                                        var startBtn = document.getElementById('start-btn');
                                        if (startBtn) {
                                            startBtn.disabled = false;
                                            startBtn.classList.remove('btn-disabled');
                                            startBtn.setAttribute('hx-post', '/start_task');
                                            htmx.process(startBtn);
                                            console.log('[CLIENT] Re-enable button.');
                                        }
                                        // Close the SSE connection
                                        var progressDisplay = document.getElementById('progress-display');
                                        if (progressDisplay && progressDisplay.sseEventSource) {
                                            console.log('[CLIENT] Closing SSE EventSource');
                                            progressDisplay.sseEventSource.close();
                                        }
                                    }, 100);
                                """)
                                yield sse_message(close_script, event='close')
                                break  # Exit after sending completion
                            else:
                                # Progress update
                                progress_value = progress_data.get('progress', 0)
                                
                                # Build status text from bar info
                                status_text = f"Processing... {progress_value:.1f}%"
                                if progress_data.get('bars'):
                                    first_bar = next(iter(progress_data['bars'].values()))
                                    status_text = f"{first_bar['desc']}: {first_bar['pct']:.1f}% ({first_bar['cur']}/{first_bar['tot']})"
                                
                                html_content = Div(
                                    create_progress_label("Progress:", id="progress-label"),
                                    create_progress_bar(value=str(int(progress_value)), id="progress-bar"),
                                    create_status_message(status_text, id="status-text")
                                )
                                
                                html_output = f"data: {to_xml(html_content)}\n\n"
                                yield sse_message(html_content)
                    except json.JSONDecodeError as e:
                        print(f"[STREAM] JSON decode error: {e}")
                        pass  # Skip invalid data
                elif data.startswith('event: end'):
                    # Handle end event - job not found or timed out waiting
                    print(f"[STREAM] Received 'event: end' - job not found or timed out")
                    
                    # Send error message
                    error_html = Div(
                        create_progress_label("Error:", id="progress-label"),
                        create_status_message("Job not found or timed out", id="status-text")
                    )
                    yield sse_message(error_html)
                    
                    # Send close event
                    close_script = Script("""
                        console.log('[CLIENT] Job not found, closing SSE');
                        setTimeout(function() {
                            // Re-enable the start button
                            var startBtn = document.getElementById('start-btn');
                            if (startBtn) {
                                startBtn.disabled = false;
                                startBtn.classList.remove('btn-disabled');
                                console.log('[CLIENT] Re-enable button.');
                                startBtn.setAttribute('hx-post', '/start_task');
                                htmx.process(startBtn);
                            }
                            // Close the SSE connection
                            var progressDisplay = document.getElementById('progress-display');
                            if (progressDisplay && progressDisplay.sseEventSource) {
                                progressDisplay.sseEventSource.close();
                            }
                        }, 100);
                    """)
                    yield sse_message(None, event='close')
                    print(f"[STREAM] Sent close event due to end event")
                    break
                elif data.startswith(': '):
                    # Heartbeat/keep-alive - pass through as-is
                    print(f"[STREAM] Sending heartbeat")
                    yield data
                    
        except Exception as e:
            print(f"[STREAM] ERROR in html_stream: {e}")
            import traceback
            traceback.print_exc()
            
            # Send error message
            error_html = Div(
                create_progress_label("Error:", id="progress-label"),
                create_status_message(f"Stream error: {str(e)}", id="status-text")
            )
            yield sse_message(error_html)
            
            # Send close event
            close_script = Script("""
                console.log('[CLIENT] Stream error, closing SSE');
                setTimeout(function() {
                    // Re-enable the start button
                    var startBtn = document.getElementById('start-btn');
                    if (startBtn) {
                        startBtn.disabled = false;
                        startBtn.classList.remove('btn-disabled');
                        console.log('[CLIENT] Re-enable button.');
                        startBtn.setAttribute('hx-post', '/start_task');
                        htmx.process(startBtn);
                    }
                    // Close the SSE connection
                    var progressDisplay = document.getElementById('progress-display');
                    if (progressDisplay && progressDisplay.sseEventSource) {
                        progressDisplay.sseEventSource.close();
                    }
                }, 100);
            """)
            yield sse_message(close_script, event='close')
        
        
    try:
        return EventStream(html_stream())
    except Exception as e:
        print(f"[STREAM] ERROR creating EventStream: {e}")
        import traceback
        traceback.print_exc()
        raise

In [11]:
# Start server for Jupyter
server = start_test_server(app, port=5003)
display(HTMX(port=server.port))

[START_TASK] Starting job with ID: be011118-4e18-4cc7-89e7-865559997852
[START_TASK] Job be011118-4e18-4cc7-89e7-865559997852 started, returning SSE progress display
[CREATE_SSE] Creating SSE progress display for job_id: be011118-4e18-4cc7-89e7-865559997852
[CREATE_SSE] SSE attributes: sse-connect=/stream?job_id=be011118-4e18-4cc7-89e7-865559997852


Processing: 100%|███████████████████████████████████████████████████████| 100/100 [00:01<00:00, 99.03it/s]


In [10]:
# Stop server when done
server.stop()